# 04 — OSM Transmission Line Topology

**Purpose:** Pull high-voltage transmission lines from OpenStreetMap via the
Overpass API and add them as a spatial backdrop to the BA flow map.

OSM data is crowdsourced and not authoritative, but it provides good spatial
coverage of the bulk transmission network (115 kV+) and is openly accessible.
It serves as the 'connective tissue' layer between BA territories — showing
the physical infrastructure that carries the flows measured in 03b.

**Note on data restrictions:** The HIFLD transmission line dataset was
restricted from public access circa 2018-2019 on FERC security grounds.
OSM is the best openly available substitute. For research requiring
authoritative topology, contact WECC/MISO/PJM/CAISO directly — each ISO
publishes its own transmission maps to varying degrees of completeness.

**Inputs:**
- OpenStreetMap Overpass API (fetched here)
- `data/processed/ba_territories.geojson` — bounding box reference

**Outputs:**
- `data/processed/osm_transmission_hv.geojson` — lines ≥115 kV, EPSG:4326
- `data/processed/combined_map.html` — BA flow map + transmission lines overlay

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import requests
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import folium
from tqdm import tqdm
import utils

In [2]:
# ── Overpass query parameters ─────────────────────────────────────────────────
# Query: all power lines tagged as 'line' or 'cable' with voltage ≥ 115000 V
# Bounding box covers the contiguous US
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'

# Voltage threshold in Volts (OSM stores voltage as a string like '345000')
MIN_VOLTAGE_KV = 115

OVERPASS_QUERY = """
[out:json][timeout:180];
(
  way["power"="line"]["voltage"~"^[0-9]+$"](24,-125,50,-66);
  way["power"="cable"]["voltage"~"^[0-9]+$"](24,-125,50,-66);
);
out geom;
"""

print('Overpass query ready. This fetch may take 60-180 seconds.')

Overpass query ready. This fetch may take 60-180 seconds.


## 1. Fetch OSM Transmission Lines

The Overpass API returns a JSON response containing OSM way objects.
Each way has a `geometry` field with the list of lat/lon nodes that
define the line, and a `tags` field with attributes including voltage.

The full CONUS query is large — expect 60-180 seconds and a response
of several hundred MB. If the query times out, reduce the bounding box
or increase `[timeout:...]`.

In [3]:
import time

# ── Check for cached result first ─────────────────────────────────────────────
osm_cache = PROJECT_ROOT / 'data' / 'raw' / 'osm_transmission_raw.json'
osm_cache.parent.mkdir(parents=True, exist_ok=True)

if osm_cache.exists():
    print(f'Loading cached OSM response from {osm_cache}')
    with open(osm_cache) as f:
        osm_data = json.load(f)
    print(f'OSM elements returned: {len(osm_data.get("elements", [])):,}')
else:
    # The full CONUS bbox times out in a single Overpass request.
    # Split into 6 regions; query each and merge elements.
    REGIONS = {
        'Northwest':   (42, -125, 50, -110),
        'Southwest':   (24, -125, 42, -110),
        'North Plains': (42, -110, 50,  -90),
        'South Plains': (24, -110, 42,  -90),
        'Northeast':   (37,  -90, 50,  -66),
        'Southeast':   (24,  -90, 37,  -66),
    }

    def overpass_query(bbox_s, bbox_w, bbox_n, bbox_e, timeout=180):
        query = f"""
[out:json][timeout:{timeout}];
(
  way["power"="line"]["voltage"~"^[0-9]+$"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
  way["power"="cable"]["voltage"~"^[0-9]+$"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
);
out geom;
"""
        for attempt in range(3):
            try:
                r = requests.post(OVERPASS_URL, data=query, timeout=timeout + 30)
                r.raise_for_status()
                return r.json().get('elements', [])
            except Exception as e:
                print(f'  Attempt {attempt+1} failed: {e}')
                time.sleep(10)
        return []

    all_elements = []
    for name, (s, w, n, e) in REGIONS.items():
        print(f'Querying {name} ({s},{w} → {n},{e})...', end=' ', flush=True)
        elems = overpass_query(s, w, n, e)
        print(f'{len(elems):,} elements')
        all_elements.extend(elems)
        time.sleep(5)  # be polite to the public server

    # Deduplicate by OSM id
    seen = set()
    unique_elements = []
    for elem in all_elements:
        eid = elem.get('id')
        if eid not in seen:
            seen.add(eid)
            unique_elements.append(elem)

    osm_data = {'elements': unique_elements}
    with open(osm_cache, 'w') as f:
        json.dump(osm_data, f)
    print(f'\nCached raw response → {osm_cache}')
    print(f'OSM elements returned: {len(osm_data["elements"]):,}')

Loading cached OSM response from /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/raw/osm_transmission_raw.json
OSM elements returned: 257,939


## 2. Parse OSM Ways into GeoDataFrame

In [4]:
# ── Parse OSM way elements into geometries ────────────────────────────────────
rows = []
for elem in tqdm(osm_data.get('elements', []), desc='Parsing OSM ways'):
    if elem.get('type') != 'way':
        continue
    geom_nodes = elem.get('geometry', [])
    if len(geom_nodes) < 2:
        continue

    coords = [(node['lon'], node['lat']) for node in geom_nodes]
    tags = elem.get('tags', {})

    # Parse voltage — OSM may store '345000', '345', or '345;115' (multi-voltage)
    voltage_raw = tags.get('voltage', '')
    # Take the first/highest numeric value
    voltages = []
    for v in voltage_raw.replace(';', ' ').split():
        try:
            voltages.append(int(v))
        except ValueError:
            pass
    if not voltages:
        continue
    voltage_v = max(voltages)  # in Volts; OSM is inconsistent — may be kV or V
    # Normalise to kV: if value > 1000 assume Volts, else assume already kV
    voltage_kv = voltage_v / 1000 if voltage_v > 1000 else voltage_v

    rows.append({
        'osm_id': elem.get('id'),
        'voltage_kv': voltage_kv,
        'name': tags.get('name', ''),
        'operator': tags.get('operator', ''),
        'cables': tags.get('cables', ''),
        'geometry': LineString(coords),
    })

print(f'Parsed {len(rows):,} way geometries')

Parsing OSM ways: 100%|██████████| 257939/257939 [00:07<00:00, 35819.19it/s]

Parsed 257,939 way geometries


In [5]:
# ── Build GeoDataFrame and filter by voltage ──────────────────────────────────
osm_gdf = gpd.GeoDataFrame(rows, crs='EPSG:4326')
print(f'Before voltage filter: {len(osm_gdf):,}')

osm_hv = osm_gdf[osm_gdf['voltage_kv'] >= MIN_VOLTAGE_KV].copy()
print(f'After filter (≥{MIN_VOLTAGE_KV} kV): {len(osm_hv):,}')

print('\nVoltage distribution (kV):')
print(osm_hv['voltage_kv'].value_counts().sort_index(ascending=False).head(15).to_string())

Before voltage filter: 257,939
After filter (≥115 kV): 168,025

Voltage distribution (kV):
voltage_kv
1333.000       1
1150.003       3
1100.000       1
765.000      379
735.000      214
690.000       76
660.000        1
600.000       14
575.000       17
500.000     5494
480.000       13
450.000       12
400.000      212
380.000        1
360.000        4


In [6]:
# ── Save HV transmission lines ────────────────────────────────────────────────
utils.save_processed(osm_hv, 'osm_transmission_hv.geojson')

Saved 168,025 features → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/osm_transmission_hv.geojson


PosixPath('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/osm_transmission_hv.geojson')

## 3. Combined Map: BA Flow + Transmission Lines

In [7]:
# ── Load prior outputs ────────────────────────────────────────────────────────
ba = gpd.read_file(PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson')
flow_summary = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'ba_interchange_summary.csv')

In [8]:
# ── Build combined map ────────────────────────────────────────────────────────
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

# Layer 1: BA territory outlines
folium.GeoJson(
    ba.__geo_interface__,
    name='BA Territories',
    style_function=lambda _: {
        'fillColor': 'transparent',
        'color': '#4466aa',
        'weight': 0.8,
        'opacity': 0.6,
    },
    tooltip=folium.GeoJsonTooltip(fields=['ba_code', 'ba_name'], aliases=['BA', 'Name'])
).add_to(m)

# Layer 2: OSM HV transmission lines — styled by voltage tier
def voltage_color(kv):
    if kv >= 500: return '#cc2222'
    if kv >= 345: return '#cc6622'
    if kv >= 230: return '#888800'
    return '#448844'

print('Adding transmission lines...')
for _, row in tqdm(osm_hv.iterrows(), total=len(osm_hv), desc='OSM lines'):
    if row.geometry is None:
        continue
    coords = [[pt[1], pt[0]] for pt in row.geometry.coords]
    folium.PolyLine(
        locations=coords,
        color=voltage_color(row['voltage_kv']),
        weight=0.6,
        opacity=0.5,
        tooltip=f"{row['voltage_kv']:.0f} kV"
    ).add_to(m)

folium.LayerControl().add_to(m)

map_path = PROJECT_ROOT / 'data' / 'processed' / 'combined_map.html'
m.save(str(map_path))
print(f'Map saved → {map_path}')

Adding transmission lines...


OSM lines: 100%|██████████| 168025/168025 [00:20<00:00, 8078.30it/s] 


Map saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/combined_map.html
